In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

## Initialise input widgets and fetch parameters

In [0]:
dbutils.widgets.text("catalog_name", "spotify_catalog")
dbutils.widgets.text("adls_storage_container_name", "")
dbutils.widgets.text("source_schema_name", "")
dbutils.widgets.text("source_table_name", "")
dbutils.widgets.text("pk_list", "")
dbutils.widgets.text("target_schema_name", "")
dbutils.widgets.text("target_table_name", "")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
adls_storage_container_name = dbutils.widgets.get("adls_storage_container_name")
source_table_schema = dbutils.widgets.get("source_schema_name")
source_table_name = dbutils.widgets.get("source_table_name")
pk_list = dbutils.widgets.get("pk_list").split(",")
target_table_schema = dbutils.widgets.get("target_schema_name")
target_table_name = dbutils.widgets.get("target_table_name")

In [0]:
import os
import sys

project_pth = os.path.join(os.getcwd(), '..', '..')
sys.path.append(project_pth)

In [0]:
from utils.transformations import reusable_transformations

transform_obj = reusable_transformations()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## SCD Type 2 table creation

In [0]:
gold_table_sk = f"{target_table_name}_sk"

In [0]:
transform_obj.create_scd2_table(
    spark,
    catalog_name,
    source_table_schema,
    source_table_name,
    target_table_schema,
    target_table_name,
    adls_storage_container_name
    )

In [0]:
target_full_table_name = f"{catalog_name}.{target_table_schema}.{target_table_name}"
source_full_table_name = f"{catalog_name}.{source_table_schema}.{source_table_name}"

## Fetch incremental records from silver layer for INSERT operation

In [0]:
source_df = spark.table(source_full_table_name)

max_ts = spark.sql(f"""
            SELECT COALESCE(MAX(ingested_at), TIMESTAMP('1900-01-01'))
            FROM {target_full_table_name}
        """).collect()[0][0]

incremental_df = source_df.filter(F.col("ingested_at") > F.lit(max_ts))

incremental_df.display()

## Append incremental records to target delta table

In [0]:
incremental_df = incremental_df \
            .withColumn("is_current", F.lit(True)) \
            .withColumn("ingested_at", F.current_timestamp()) \
            .withColumn("active_start_date_time", F.current_timestamp()) \
            .withColumn("active_end_date_time", F.lit(None).cast("timestamp"))

In [0]:
incremental_df.write.format("delta").mode("append").saveAsTable(target_full_table_name)

## Invalidate/expire old records from SCD Type2 table

In [0]:
mart_df = spark.table(target_full_table_name)

In [0]:
mart_latest_records_window_spec = Window.partitionBy(pk_list).orderBy(F.desc("active_start_date_time"), F.desc("ingested_at"))

mart_iscurrent_record_df = mart_df.withColumn("rn", F.row_number().over(mart_latest_records_window_spec)) \
    .filter((F.col("is_current") == True) & (F.col("active_end_date_time").isNull())) \
    .filter("rn=1")

mart_iscurrent_record_df.display()

In [0]:
# Dynamic join condition
dynamic_join_condition = " AND ".join([f"t.{c} = s.{c}" for c in pk_list])
static_join_condition = f"AND t.{gold_table_sk} != s.{gold_table_sk}"

final_join_condition = f"{dynamic_join_condition} {static_join_condition}"
print(f"Final join condition is: {final_join_condition}")

In [0]:
mart_df.alias("t").merge(
            mart_iscurrent_record_df.alias("s"),
            dynamic_join_condition
        ).whenMatchedUpdate(
            condition="""
                t.is_current = true 
                AND t.active_end_date_time is null
            """,
            set={
                "is_current": "false",
                "active_end_date_time": "current_timestamp()"
            }
        ).execute()